# Day 81: Multivariable regression and valuation model

You are an analyst at a real estate development company in Boston in the 1970s. The company wants a price estimate for a residential project, based on the number of rooms, the distance to employment centres, how poor the area is, the number of students per teacher and similar characteristics. This notebook follows the project's seven steps and calls the functions of the `boston_valuation` package.

Install the notebook extra first: `python -m pip install -e ".[notebook]"`.

# Step 1: Goals and data exploration

In [ ]:
from boston_valuation.domain import descriptive, exploration
from boston_valuation.infrastructure.csv_loader import load_data
from boston_valuation.presentation.charts import descriptive_table_figure

data = load_data()
data.head()

## Preliminary exploration

How many rows and columns are there, what are the columns called, and is there anything missing or duplicated?

In [ ]:
print("Shape:", exploration.table_shape(data))
print("Columns:", exploration.column_names(data))
print("NaN values:", exploration.has_nan_values(data))
print("Duplicates:", exploration.has_duplicates(data))

In [ ]:
exploration.first_rows(data)

In [ ]:
exploration.last_rows(data)

## Descriptive statistics

The average number of students per teacher, the average price (in thousands of dollars), the two values of the Charles River dummy `CHAS` and the range of rooms per dwelling.

In [ ]:
print("Students per teacher:", round(descriptive.average_students_per_teacher(data), 2))
print("Average price ($1000s):", round(descriptive.average_price(data), 2))
print("Average price ($):", round(descriptive.average_price_in_dollars(data)))
print("CHAS min and max:", descriptive.river_value_range(data))
print("Tracts away from / next to the river:")
print(descriptive.river_counts(data))
print("Rooms min and max:", descriptive.room_range(data))

In [ ]:
descriptive.descriptive_statistics(data)

In [ ]:
figure = descriptive_table_figure(data)
figure

# Step 2: Feature distributions

Look at one feature at a time: the shape of the price, the distance to the employment centres, the rooms and the highway access, and how many homes border the Charles River.

In [ ]:
from boston_valuation.presentation.distribution_charts import (
    distance_distribution_figure,
    highway_access_figure,
    price_distribution_figure,
    river_bar_chart,
    rooms_distribution_figure,
)

price_distribution_figure(data)

In [ ]:
distance_distribution_figure(data)

In [ ]:
rooms_distribution_figure(data)

In [ ]:
highway_access_figure(data)

`RAD` only takes the values 1 to 8 and 24, so the bars have gaps. The river dummy is a bar chart of two counts.

In [ ]:
river_bar_chart(data)

# Step 3: Relationships in the data

Before modelling, predict what you expect: does pollution (`NOX`) rise or fall with the distance to employment (`DIS`)? What do the rooms (`RM`) and the poverty of an area (`LSTAT`) do to the price? Then check. A pair plot of all 14 columns can take a minute or two, so this notebook draws it for four columns.

In [ ]:
from boston_valuation.domain import relationships
from boston_valuation.presentation.relationship_charts import (
    distance_pollution_figure,
    pair_plot_figure,
    pollution_industry_figure,
    poverty_price_figure,
    poverty_rooms_figure,
    rooms_price_figure,
)

pair_plot_figure(data, columns=["RM", "LSTAT", "NOX", "PRICE"], regression_line=True)

Pass `columns=None` to draw the pair plot of every column, as in the course.

In [ ]:
distance_pollution_figure(data)

In [ ]:
pollution_industry_figure(data)

In [ ]:
poverty_rooms_figure(data)

In [ ]:
poverty_price_figure(data)

In [ ]:
rooms_price_figure(data)

Each observation is backed by a correlation coefficient.

In [ ]:
print("DIS vs NOX:", round(relationships.distance_pollution_correlation(data), 2))
print("NOX vs INDUS:", round(relationships.pollution_industry_correlation(data), 2))
print("LSTAT vs RM:", round(relationships.poverty_rooms_correlation(data), 2))
print("LSTAT vs PRICE:", round(relationships.poverty_price_correlation(data), 2))
print("RM vs PRICE:", round(relationships.rooms_price_correlation(data), 2))
relationships.correlation_with_price(data)

# Step 4: Split and multivariable regression

Set 20 % of the homes aside (`random_state=10`), fit a linear regression on the other 80 % and read the coefficients and the residuals.

In [ ]:
from boston_valuation.domain import regression
from boston_valuation.presentation.regression_charts import (
    actual_vs_predicted_figure,
    residual_distribution_figure,
    residuals_vs_predicted_figure,
)

features, target = regression.split_features_and_target(data)
split = regression.split_train_test(features, target)
train_pct, test_pct = regression.split_percentages(split)
print(f"Training data is {train_pct:.3}% of the total data.")
print(f"Test data makes up the remaining {test_pct:.3}%.")

In [ ]:
regr = regression.fit_linear_regression(split.x_train, split.y_train)
rsquared = regression.r_squared(regr, split.x_train, split.y_train)
print(f"Training data r-squared: {rsquared:.2}")

In [ ]:
regr_coef = regression.coefficient_table(regr, split.x_train.columns)
regr_coef

In [ ]:
premium = regression.room_premium(regr_coef)
print(f"The price premium for having an extra room is ${premium:.5}")

In [ ]:
predicted_vals = regression.predictions(regr, split.x_train)
residuals = regression.residuals(split.y_train, predicted_vals)
actual_vs_predicted_figure(split.y_train, predicted_vals)

In [ ]:
residuals_vs_predicted_figure(predicted_vals, residuals)

In [ ]:
print("Mean:", round(regression.residual_mean(residuals), 2), "Skew:", round(regression.residual_skew(residuals), 2))
residual_distribution_figure(residuals)

# Step 5: The log transformation

The price is skewed to the right and so are the residuals of the first model. Take the natural logarithm of the price, fit the same regression with the same split and compare the two models, first on the training data and then on the test data.

In [ ]:
from boston_valuation.domain import log_model
from boston_valuation.presentation.log_charts import (
    log_actual_vs_predicted_figure,
    log_mapping_figure,
    log_price_skew_figure,
    log_residual_distribution_figure,
    log_residuals_vs_predicted_figure,
    model_comparison_panel_figure,
    price_skew_figure,
)

print("Skew of the price:", round(log_model.price_skew(data), 2))
print("Skew of the log price:", round(log_model.log_price_skew(data), 2))
price_skew_figure(data)

In [ ]:
log_price_skew_figure(data)

In [ ]:
log_mapping_figure(data)

In [ ]:
linear = log_model.fit_linear_model(data)
log_fit = log_model.fit_log_model(data)
print(f"Linear model, training r-squared: {linear.train_r_squared:.2}")
print(f"Log model, training r-squared: {log_fit.train_r_squared:.2}")

In [ ]:
log_coef = log_model.log_coefficient_table(log_fit)
print("Expected signs:", log_model.has_expected_signs(log_coef))
log_coef

In [ ]:
log_actual_vs_predicted_figure(data)

In [ ]:
log_residuals_vs_predicted_figure(data)

In [ ]:
log_residual_distribution_figure(data)

Out of sample: how do the two models score on the homes they have not seen?

In [ ]:
scores = log_model.compare_test_r_squared(data)
print(f"Original model test data r-squared: {scores.linear:.2}")
print(f"Log model test data r-squared: {scores.log:.2}")

In [ ]:
model_comparison_panel_figure(data)